# RAG_time

Part 6's advisor got real information two ways: calling a live tool (`imt_taf_list`) or stuffing an entire researched briefing straight into its instructions (Program 14's `briefing_text`, built once for *every* TAF, whether the student ever asks about them or not). That works at ten TAF. It stops working once the source material is a hundred-page manual, a whole wiki, or a folder of PDFs -- Part 3's context window is finite, and most of that text would be irrelevant to any single question anyway.

**RAG** (Retrieval-Augmented Generation) is the fix: embed the source material once, in advance (Part 4's embeddings, just applied to whole chunks of text instead of single tokens), embed the question the same way, and use cosine similarity to pull out only the few chunks actually relevant to it -- then hand *those* to the model, instead of everything. Let's build one, end to end, using the same real TAF data as Part 6.

## From token embeddings to sentence embeddings

Part 4 embedded single tokens. RAG needs to embed whole chunks of text -- sentences, paragraphs -- so they can be compared to a whole question. A simple, homemade way to get there with any model: run the text through it, and average ("mean-pool") the resulting per-token vectors into a single one.

In [ ]:
# Program 1: a homemade sentence embedding, by mean-pooling token vectors

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoModel (not AutoModelForCausalLM, Part 3's choice) gives direct access to hidden
# states, without the extra layer that predicts next-token logits -- we don't need that here.
base_model = AutoModel.from_pretrained(model_name)
base_model.eval()

def sentence_embedding(text):
    """A simple sentence embedding: the average of all its tokens' final hidden states."""
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        hidden_states = base_model(**inputs).last_hidden_state  # shape: (1, num_tokens, embedding_dim)
    return hidden_states.mean(dim=1).squeeze(0)  # average over the tokens -> a single vector

sentences = [
    "The cat sat on the mat.",
    "A feline was resting on the rug.",
    "The stock market crashed yesterday.",
]

embeddings = [sentence_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

The two sentences that mean roughly the same thing (the cat/feline ones) score noticeably higher than either has with the unrelated sentence about the stock market -- even though they don't share a single word. That's the whole point of a sentence embedding: it captures meaning, not vocabulary overlap.

## A real embeddings API

Mean-pooling a small local model's hidden states works, but it's a rough approximation -- `SmolLM2` was never specifically trained to produce good sentence embeddings. Providers instead offer dedicated **embedding models**, trained precisely for this. Like everywhere else in this course, we can call one through the same OpenAI-compatible client, just changing the endpoint: `embeddings.create` instead of `chat.completions.create`. Let's use Gemini's, on the same three sentences.

In [ ]:
# Program 2: sentence embeddings via a real embeddings API (Gemini)

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

def gemini_embedding(text):
    response = gemini.embeddings.create(model="gemini-embedding-001", input=text)
    return torch.tensor(response.data[0].embedding)

gemini_embeddings = [gemini_embedding(s) for s in sentences]

for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = F.cosine_similarity(gemini_embeddings[i].unsqueeze(0), gemini_embeddings[j].unsqueeze(0)).item()
        print(f"{similarity:.3f}  {sentences[i]!r}\n       <->  {sentences[j]!r}")

Same ranking as our homemade version, but with a clearer gap between the related pair and the unrelated one -- exactly what we'd expect from a model actually trained for this task. Note also the vector length: `gemini-embedding-001` returns 3072 numbers per sentence, regardless of how long the sentence is -- a fixed-size summary of its meaning, whether it's fed three words or three paragraphs.

## Building a real corpus to retrieve from

Let's reuse Part 6's real, current TAF list -- but this time, instead of researching every single one of them up front and handing the whole lot to an agent (Program 14's approach), we'll embed a short description of each TAF *once*, and only retrieve the one that actually matches a student's question.

In [ ]:
# Program 3: fetch the real TAF list, build a short description for each, and embed them all

import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (PLIDOagent-course/1.0; educational use)"}
TAF_PAGE_URL = "https://moodle.imt-atlantique.fr/course/view.php?id=897&section=6"

def fetch_taf_list():
    """Fetch the current list of TAF names in 'Informatique et Réseaux', straight off Moodle."""
    response = requests.get(TAF_PAGE_URL, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.text, "html.parser")
    section = soup.find("li", {"id": "section-6"})
    names = []
    for link in section.find_all("a"):
        text = link.get_text(strip=True).removesuffix("URL").strip()
        if text.upper().startswith("TAF") and text not in names:
            names.append(text)
    return names

# A short, accurate English description per TAF, keyed by a keyword from its (French) name --
# this is the "source material" we're indexing, the same role a real manual or wiki would play.
TAF_DOMAIN_KEYWORDS = {
    "développement collaboratif": "collaborative, multi-site software development, version control, and large-scale engineering practices.",
    "data science": "statistics, machine learning, and data visualization for business decision-making.",
    "cybersécurité": "network security, cryptography, penetration testing, and defending systems against attacks.",
    "internet des objets": "connected sensors, industrial automation, and cloud-based IoT platforms for Industry 4.0.",
    "ihm": "human-computer interaction, user interface design, and collaborative software systems.",
    "plateformes numériques": "cloud computing, platform economics, and digital infrastructure.",
    "mathematical and computational": "numerical methods, simulation, and applied mathematics for engineering.",
    "systèmes embarqués": "real-time operating systems, hardware-software co-design, and low-level programming.",
    "ingénierie logicielle et innovation": "software architecture, agile methods, and building new digital products.",
    "systèmes distribués": "microservices, networked applications, and large-scale distributed system design.",
}

def domain_blurb(taf_name):
    lowered = taf_name.lower()
    for keyword, blurb in TAF_DOMAIN_KEYWORDS.items():
        if keyword in lowered:
            return blurb
    return "no description available yet for this TAF."

taf_corpus = [{"name": name, "text": f"{name}: {domain_blurb(name)}"} for name in fetch_taf_list()]

print(f"Indexing {len(taf_corpus)} TAF descriptions...")
for entry in taf_corpus:
    entry["embedding"] = gemini_embedding(entry["text"])
    print(f"-- {entry['name']}")

Ten short descriptions would easily fit in a single prompt -- Program 14 proved that by doing exactly this. RAG's value shows up once the corpus is too large for that: a hundred TAFs, a thousand-page handbook, a whole company wiki. The mechanism is identical at any scale; only the size of `taf_corpus` changes. Each entry is embedded exactly *once*, no matter how many questions get asked against it afterwards.

## Retrieving the right chunk

With everything embedded, retrieval is just the cosine similarity from Part 4, run once per stored entry: embed the question, compare it to every entry's embedding, and keep the closest match.

In [ ]:
# Program 4: retrieve the single closest TAF description to a question, by embedding similarity

def retrieve(question, top_k=1):
    question_embedding = gemini_embedding(question)
    scored = [
        (F.cosine_similarity(question_embedding.unsqueeze(0), entry["embedding"].unsqueeze(0)).item(), entry)
        for entry in taf_corpus
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top_k]

questions = [
    "Which TAF should I pick if I love hacking and security?",
    "I'm interested in AI and data analysis, what do you suggest?",
    "I want to work on connected devices and factories.",
]

for question in questions:
    best_score, best_entry = retrieve(question)[0]
    print(f"Q: {question}\n-> {best_score:.3f}  {best_entry['name']}\n")

All three questions retrieve the right TAF, despite sharing barely any vocabulary with its description ("hacking and security" -> `Cybersécurité`; "AI and data analysis" -> `Data science`; "connected devices and factories" -> the IoT/Industry 4.0 TAF) -- the same meaning-over-spelling behavior Program 1 and 2 already showed, just now used to pick a document instead of to compare two.

Notice, too, *who* decided which TAF was relevant here: not our own Python code branching on keywords, and not an LLM reading the question -- a nearest-neighbor search over embeddings. That's a third kind of decision-maker, alongside the algorithmic and agentic ones Part 6 kept contrasting: relevance decided by geometry.

## Wrapping retrieval as a tool

Exactly like every other capability in this course, retrieval becomes useful to an agent once it's wrapped as a tool. This time the tool doesn't fetch a live web page or write a file -- it searches an in-memory index we built ourselves.

In [ ]:
# Program 5: a RAG agent -- retrieves only the relevant TAF, never sees the rest

import os
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI

RENNES_BASE_URL = "https://ragarenn.eskemm-numerique.fr/sso/instance@imt/api/"
rennes_client = AsyncOpenAI(base_url=RENNES_BASE_URL, api_key=os.environ["RENNES_API_KEY"])
rennes_model = OpenAIChatCompletionsModel(model="ilaas/mistral-small-4-119b", openai_client=rennes_client)

@function_tool
def retrieve_taf_info(question: str):
    """Retrieve the single TAF description most relevant to a student's question, out of the
    current TAF list, using embedding similarity -- not the whole list, just the best match."""
    best_score, best_entry = retrieve(question)[0]
    return best_entry["text"]

rag_agent = Agent(
    name="RAG TAF Advisor",
    instructions="Answer the student's question about TAF programs using the retrieve_taf_info "
                 "tool -- never answer from memory alone, and never mention a TAF the tool "
                 "didn't return.",
    model=rennes_model,
    tools=[retrieve_taf_info],
)

result = await Runner.run(rag_agent, "I love hacking and breaking into systems (ethically!). Which TAF fits me?")
print(result.final_output)

The agent correctly names `TAF Cybersécurité` -- but notice what it *couldn't* have done: it never saw the other nine TAF descriptions at all, only whatever `retrieve_taf_info` handed back. Compare that to Program 14, which researched and sent *every* TAF's briefing on every single call, whether the conversation needed it or not. RAG is the same "give the agent real information" idea from Part 6, just scoped down to the one piece of it that actually matters for the question being asked.

## Key takeaways

* A **sentence embedding** extends token embeddings (Part 4) to whole chunks of text -- by mean-pooling a local model's hidden states, or via a dedicated embeddings API -- giving a fixed-size vector regardless of length.
* **RAG** (Retrieval-Augmented Generation) means embedding source material once, in advance, then using cosine similarity between the question's embedding and each chunk's to retrieve only what's relevant -- instead of stuffing everything into every prompt, which is what Part 6's Program 14 did instead.
* At ten short TAF descriptions, stuffing everything in still works fine -- RAG's advantage only shows up once the source material is too large to fit, or too wasteful to resend on every call. The mechanism doesn't change with scale, only the size of the corpus does.
* Retrieval is a third kind of decision-maker alongside Part 6's algorithmic and agentic ones: relevance is decided by nearest-neighbor search over embeddings, not by fixed Python branches and not by an LLM reading everything.
* Wrapped as a `@function_tool`, a retrieval step looks exactly like any other tool to the agent -- the same pattern used for `imt_taf_list`, `domain_lookup`, or `brave_search_tool` in Part 6.

## Function reference

A quick reference for the less obvious functions and methods used in this notebook (skipping ones already familiar from earlier parts, like `AutoTokenizer.from_pretrained`, `Agent`, `Runner.run`, and `@function_tool`):

| Function (module) | Arguments | Returns | Used in |
|---|---|---|---|
| `AutoModel.from_pretrained(name)` (`transformers`) | a model name | a model exposing raw hidden states, no next-token head | Program 1 |
| `model(**inputs).last_hidden_state` (`transformers`) | tokenized input | one vector per input token | Program 1 |
| `tensor.mean(dim=1)` (`torch`) | a dimension to average over | the tensor with that dimension collapsed | Program 1 |
| `gemini.embeddings.create(model=, input=)` (`openai`) | a model name, a string | an embedding response (`.data[0].embedding`) | Programs 2, 3, 4 |
| `F.cosine_similarity(a, b)` (`torch.nn.functional`) | two tensors of the same shape | their cosine similarity | Programs 1, 2, 4 |